# Data fetching and processing

This notebook includes the data fetching and processing into files used by the analysis and visualisation in this project. The file is authored by Michał Jan Odorczuk and strongly (and shamelessly) bases on the code generated by ChatGPT.

Consult other notebooks for the full research process. The available notebooks are:
 - [Main notebook](main.ipynb) - notebook including the introduction of the problem and explanation of the available data and the area of interest.
 - [Data fetching notebook](data_fetching.ipynb) - notebook including the whole pipeline for fetching the data from the Copernicus database, filtering it, processing, and storing in the format used by the further analysis.
 - [Local maps notebook](local_maps.ipynb) - demo notebook showing examples of available data for the areas of interest in the form of maps of small regions within our area of interest.
 - [Profile notebook](profile.ipynb) - demo notebook showing examples of available data for the whole river in the form of spectral (and spectra-derived) characteristics along the Oder centreline.

## Used libraries

In [1]:
from pathlib import Path
import geopandas as gpd
from scripts.copernicus_api import get_access_token, search_tile, download_product
from scripts.print_aux import progress_bar
import os
import re
import json5 as json
from scripts.file_aux import extract_bands_from_zip, process_resolution_for_aoi, sample_points, read_parameters, downsample

## Metadata fetching

In [2]:
parameters = read_parameters("metadata.json")
tiles = parameters["tiles"]
dates = parameters["dates"]
MAP_CRS = parameters["MAP_CRS"]
aoi_dir = Path(parameters["objects"]["small_AOI"])
aoi_polygons = gpd.read_file(aoi_dir).to_crs(MAP_CRS)
centreline_dir = Path(parameters["objects"]["Oder_centreline"])
oder_centreline = gpd.read_file(centreline_dir).to_crs(MAP_CRS)
oder_geometry = oder_centreline.geometry.iloc[0]

## Download data from [Copernicus](https://browser.dataspace.copernicus.eu/) dataset

In [3]:
download_dir = Path(parameters["data_directories"]["raw_data"])

print("Requesting Copernicus access token...")
token = get_access_token(parameters["API"]["SentinelHub"]["client_id"], parameters["API"]["SentinelHub"]["client_secret"])

all_products = []
tiles_n = 0
for date in dates:
    tiles_n += len(tiles)
tile_i = 0
for date in dates:
    for tile in tiles:
        tile_i += 1

        progress_bar(tile_i, tiles_n, bar_length=50)
        print(f"\nSearching {tile_i}/{tiles_n} tile {tile}...")

        products = search_tile(tile, token, date)

        if not products:
            print("  No matching Sentinel-2 L2A products found.")
            continue

        for product in products:

            print(
                f"  Found: {product['Name']}\n"
                f"  Sensing start: {product['ContentDate']['Start']}\n"
                f"  Product ID: {product['Id']}\n"
            )

            all_products.append(product)

unique_products = {
    product["Id"]: product
    for product in all_products
}

print(f"\nTotal unique products found: {len(unique_products)}")

for product in unique_products.values():
    download_product(product, token, download_dir)

print("\nFinished.")

Requesting Copernicus access token...
[>                                                 ] 1% (1/84)

Searching 1/84 tile T33UVA...
  Found: S2A_MSIL2A_20220816T100611_N0510_R022_T33UVA_20240710T182750.SAFE
  Sensing start: 2022-08-16T10:06:11.024000Z
  Product ID: c39598f2-b677-4e81-974a-f4bd1920f45b

[>                                                 ] 2% (2/84)

Searching 2/84 tile T33UVV...
  Found: S2A_MSIL2A_20220816T100611_N0510_R022_T33UVV_20240710T182750.SAFE
  Sensing start: 2022-08-16T10:06:11.024000Z
  Product ID: 70dab37a-1a93-4c08-a2a2-11e696056757

[>                                                 ] 3% (3/84)

Searching 3/84 tile T33UVU...
  Found: S2A_MSIL2A_20220816T100611_N0510_R022_T33UVU_20240710T182750.SAFE
  Sensing start: 2022-08-16T10:06:11.024000Z
  Product ID: 59e14319-facb-4cb1-8a91-d2b9c7ad0063

[=>                                                ] 4% (4/84)

Searching 4/84 tile T33UVT...
  Found: S2A_MSIL2A_20220816T100611_N0510_R022_T33UVT_20240710T182750.

## Extracting bands

We are interested only in 10 and 20m bands. 

In [4]:
files = os.listdir(download_dir)

zips = []
bands10 = parameters["bands"]["short_names"]["10m"]
bands20 = parameters["bands"]["short_names"]["20m"]
full_bands = [b10 + "_10m" for b10 in bands10] + [b20 + "_20m" for b20 in bands20]
bands = [b for b in full_bands if b != "SCL_20m"]
band_wavelengths = parameters["bands"]["wavelengths"]
bands = sorted(bands, key=lambda band: band_wavelengths[band[:-4]])
band_names = [b[:-4] + " " + parameters["bands"]["descriptions"][b[:-4]] for b in bands]
wavelengths = [band_wavelengths[band[:-4]] for band in bands]

times = {}

for file in files:
    match = re.match(
        r"S2\w_MSIL2A_(\d+)T(\d+)_(?:N\d+)_(?:R\d+)_(T\d\d\w\w\w)_(?:\d+T\d+)\.SAFE\.zip",
        file)
    if match:
        date, time, tile = match.group(1), match.group(2), match.group(3)
        zips.append((date, tile, file))
        if not date in times:
            times[date] = {}
        times[date][tile] = time

json.dump(times, open(Path(parameters["data_directories"]["raw_data"]) / "sample_times.json", "w"), indent=4)

file_i = 0
for (date, tile, filename) in zips:
    file_i += 1
    extract_bands_from_zip(date, tile, filename, bands10, bands20)
    progress_bar(file_i, len(zips), bar_length=50)


Extracted directory already exists for 20220212_T33UVU. Skipping extraction.
[>                                                 ] 1% (1/64)
Extracted directory already exists for 20220212_T33UVA. Skipping extraction.
[>                                                 ] 3% (2/64)
Extracted directory already exists for 20220719_T33UYR. Skipping extraction.
[=>                                                ] 4% (3/64)
Extracted directory already exists for 20220805_T33UXS. Skipping extraction.
[==>                                               ] 6% (4/64)
Extracted directory already exists for 20220816_T33UVV. Skipping extraction.
[==>                                               ] 7% (5/64)
Extracted directory already exists for 20200811_T33UVV. Skipping extraction.
[===>                                              ] 9% (6/64)
Extracted directory already exists for 20220719_T33UVT. Skipping extraction.
[====>                                             ] 10% (7/64)
Extracted directory

In [5]:
aoi_dir = parameters["objects"]["small_AOI"]
band_dir = parameters["data_directories"]["extracted"]

nodata_value = 0

data_dirs = os.listdir(band_dir)
data_by_dates = {}
band_files = {}
band_i = 0
for dir in data_dirs:
    date = dir.split("_")[0]
    if date not in data_by_dates:
        data_by_dates[date] = []
        band_files[date] = {band.split("_")[0]: [] for band in full_bands}
    data_by_dates[date].append(dir)
    for band in full_bands:
        band_i += 1
        progress_bar(band_i, len(data_dirs) * len(full_bands), bar_length=50)
        band_file = Path(band_dir) / dir / f"R{band[-3:]}" / f"{band}.jp2"
        if band_file.exists():
            band_files[date][band.split("_")[0]].append(band_file)

[>                                                 ] 0% (1/704)
[>                                                 ] 0% (2/704)
[>                                                 ] 0% (3/704)
[>                                                 ] 0% (4/704)
[>                                                 ] 0% (5/704)
[>                                                 ] 0% (6/704)
[>                                                 ] 0% (7/704)
[>                                                 ] 1% (8/704)
[>                                                 ] 1% (9/704)
[>                                                 ] 1% (10/704)
[>                                                 ] 1% (11/704)
[>                                                 ] 1% (12/704)
[>                                                 ] 1% (13/704)
[>                                                 ] 1% (14/704)
[>                                                 ] 2% (15/704)
[>                                

## Mask and save the areas of interest

In [6]:
masked_dir = Path(parameters["data_directories"]["masked"])
masked_dir.mkdir(parents=True, exist_ok=True)

aois = aoi_polygons.dissolve(by="NAME", as_index=False)

aoi_n = 0
for date_text, tile_folders in sorted(data_by_dates.items()):
    aoi_n += len(aois)

aoi_i = 0
for date_text, tile_folders in sorted(data_by_dates.items()):
    for _, aoi_row in aois.iterrows():
        aoi_i += 1
        aoi_name = aoi_row["NAME"]
        aoi_geometry = aoi_row.geometry
        process_resolution_for_aoi(
            date_text=date_text,
            band_files=band_files,
            aoi_name=aoi_name,
            aoi_geometry=aoi_geometry,
            resolution=10,
            bands=bands10,
            output_root=masked_dir,
            target_crs=MAP_CRS
        )

        process_resolution_for_aoi(
            date_text=date_text,
            band_files=band_files,
            aoi_name=aoi_name,
            aoi_geometry=aoi_geometry,
            resolution=20,
            bands=bands20,
            output_root=masked_dir,
            target_crs=MAP_CRS
        )
        progress_bar(aoi_i, aoi_n, bar_length=50)

Stack file already exists for AOI Lower_Oder and resolution 10m. Skipping processing.
Stack file already exists for AOI Lower_Oder and resolution 20m. Skipping processing.
[>                                                 ] 0% (1/216)
Stack file already exists for AOI Oder_1 and resolution 10m. Skipping processing.
Stack file already exists for AOI Oder_1 and resolution 20m. Skipping processing.
[>                                                 ] 0% (2/216)
[>                                                 ] 1% (3/216)
[>                                                 ] 1% (4/216)
[>                                                 ] 2% (5/216)
[>                                                 ] 2% (6/216)
Stack file already exists for AOI Oder_2 and resolution 10m. Skipping processing.
Stack file already exists for AOI Oder_2 and resolution 20m. Skipping processing.
[>                                                 ] 3% (7/216)
Stack file already exists for AOI Oder_3 and resolut

## Downscale the estuary area

The display of large areas with too fine resolution can be quite heavy. Thus, we prepare separate downscaled TIF files for Pomeranian Bay, Usedom-Wolin archipelago, and Szczecin Lagoon.

In [7]:
masked_dir = Path(parameters["data_directories"]["masked"])
downsampled_dir = Path(parameters["data_directories"]["downsampled"])
downsampled_dir.mkdir(parents=True, exist_ok=True)
tile_n = 0
for date in os.listdir(masked_dir):
    for tile in ["Pomeranian_Bay", "Usedom_Wolin", "Szczecin_Lagoon"]:
        if (masked_dir / date / tile).is_dir():
            tile_n += 1
tile_i = 0
for date in os.listdir(masked_dir):
    for tile in ["Pomeranian_Bay", "Usedom_Wolin", "Szczecin_Lagoon"]:
        if not (masked_dir / date / tile).is_dir():
            continue
        tile_i += 1
        for stack in ["10m_Stack.tif", "20m_Stack.tif"]:
            input_file = masked_dir / date / tile / stack
            output_parent = downsampled_dir / date / tile
            output_parent.mkdir(parents=True, exist_ok=True)
            downsample(input_file, output_parent / stack, resolution=100)
        progress_bar(tile_i, tile_n, bar_length=50)

[=>                                                ] 4% (1/24)
[===>                                              ] 8% (2/24)
[=====>                                            ] 12% (3/24)
[=======>                                          ] 16% (4/24)
[=========>                                        ] 20% (5/24)
[===========>                                      ] 25% (6/24)
[=============>                                    ] 29% (7/24)
[===============>                                  ] 33% (8/24)
[=================>                                ] 37% (9/24)
[===================>                              ] 41% (10/24)
[=====================>                            ] 45% (11/24)
[========================>                         ] 50% (12/24)
[==========================>                       ] 54% (13/24)
[============================>                     ] 58% (14/24)
[==============================>                   ] 62% (15/24)
[================================>  

## Sample and save profile data for all dates

In [8]:
masked_dirs = os.listdir(masked_dir)
profile_dir = Path(parameters["data_directories"]["profiles"])
profile_dir.mkdir(parents=True, exist_ok=True)
profiles = {}
SCL_WATER_CLASS = 6
res_n = len(masked_dirs) * 2
res_i = 0
for dir in masked_dirs:
    date = dir.split("/")[-1]
    for resolution in ["10m", "20m"]:
        res_i += 1
        profile = sample_points(
            data_dir=masked_dir / dir,
            line_geometry=oder_geometry,
            sample_spacing=10,
            sample_class=SCL_WATER_CLASS,
            band_names10=parameters["bands"]["short_names"]["10m"],
            band_names20=parameters["bands"]["short_names"]["20m"]
        )
        profiles[date] = profile
        profile.to_csv(profile_dir / f"{date}.csv", index=False)
        progress_bar(res_i, res_n, bar_length=50)


[=>                                                ] 4% (1/24)
[===>                                              ] 8% (2/24)
[=====>                                            ] 12% (3/24)
[=======>                                          ] 16% (4/24)
[=========>                                        ] 20% (5/24)
[===========>                                      ] 25% (6/24)
[=============>                                    ] 29% (7/24)
[===============>                                  ] 33% (8/24)
[=================>                                ] 37% (9/24)
[===================>                              ] 41% (10/24)
[=====================>                            ] 45% (11/24)
[========================>                         ] 50% (12/24)
[==========================>                       ] 54% (13/24)
[============================>                     ] 58% (14/24)
[==============================>                   ] 62% (15/24)
[================================>  